In [ ]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
import pandas as pd

# Import pour calcul des surfaces de cellules
from seapopym.transport import compute_spherical_cell_areas
from seapopym.standard.coordinates import Coordinates

print("✅ Imports loaded")

# Comparison: LMTL (C++) vs SeapoPym (Python)

This notebook compares the transport-only simulations from two models using identical synthetic forcing.

**Objective:** Verify that both models produce similar results when numerical diffusion is properly controlled.

**Configuration:**
- Domain: 25° × 25° at 1/12° resolution (~9.3 km)
- Forcing: Uniform eastward current (u=1.0 m/s)
- Physics: Advection + Diffusion (D_phys = 3000 m²/s)
- CFL ≈ 0.49 (optimized for D_phys > D_num)
- Period: 1 year (2000-2001)

**Metrics:**
- MAPE/MPE: Percentage errors (area-weighted)
- RMSE/NRMSE: Root mean square errors (area-weighted)
- Mass conservation: Total biomass evolution
- Spatial distributions: Visual comparison

In [ ]:
# Resample interval for animations (weekly aggregation)
interval = "1W"  # 1 week - good balance for 1-year simulation

# Timestep configuration (from SeapoPym simulation)
dt_hours = 1  # 1h15min = 1.25h timestep
timesteps_per_day = int(24 / dt_hours)  # 19 timesteps per day (approximately)
print(f"SeapoPym timestep: {dt_hours:.2f}h ({dt_hours * 3600:.0f}s)")
print(f"Timesteps per day: {timesteps_per_day} (will subsample to match LMTL daily output)")

In [ ]:
# Load LMTL biomass (C++ model output)
print("Loading LMTL biomass...")
lmtl_biomass = xr.open_mfdataset("./LMTL_Synthetic/output/*biomass*.nc")
lmtl_biomass = lmtl_biomass.biomass
lmtl_biomass = lmtl_biomass.rename({"longitude": "x", "latitude": "y"})

print(f"LMTL biomass loaded: {len(lmtl_biomass.time)} daily timesteps")
print(f"Period: {lmtl_biomass.time.values[0]} to {lmtl_biomass.time.values[-1]}")
lmtl_biomass.time

In [ ]:
# Load SeapoPym biomass (Python model output)
print("Loading SeapoPym biomass...")
seapopym_biomass = xr.open_dataset("./SeapoPym_Synthetic.zarr")
seapopym_biomass = seapopym_biomass.biomass

print(f"SeapoPym biomass loaded: {len(seapopym_biomass.time)} timesteps")
print(f"Timestep: {dt_hours:.2f}h ({timesteps_per_day} per day)")
print(f"Period: {seapopym_biomass.time.values[0]} to {seapopym_biomass.time.values[-1]}")
seapopym_biomass

In [ ]:
# Find optimal initial offset between LMTL and SeapoPym
# LMTL outputs daily, SeapoPym has sub-daily timesteps
# We need to align them by finding which SeapoPym timestep best matches LMTL t=0

print("Finding optimal temporal alignment...")
min_diff = 1e10
best_offset = 0

for i in range(timesteps_per_day * 2):
    diff = float(np.abs(lmtl_biomass.isel(time=0) - seapopym_biomass.isel(time=i)).max())
    print(f"  Offset {i}: max difference = {diff:.6e}")
    if diff < min_diff:
        best_offset = i
        min_diff = diff

print(f"\n✅ Best offset: {best_offset} (min difference: {min_diff:.6e})")

# Subsample SeapoPym to match LMTL daily output
seapopym_biomass = seapopym_biomass.isel(time=slice(best_offset, None, timesteps_per_day))
print(f"SeapoPym subsampled: {len(seapopym_biomass.time)} timesteps (daily)")
seapopym_biomass

In [ ]:
lmtl_biomass = lmtl_biomass.assign_coords({"time": lmtl_biomass.time.dt.floor("D")})
seapopym_biomass = seapopym_biomass.assign_coords({"time": seapopym_biomass.time.dt.floor("D")})

In [ ]:
# Load data into memory and compute cell areas
print("Loading data into memory...")
lmtl_biomass.load()
_ = seapopym_biomass.load()

# Calculate cell areas (necessary for area-weighted metrics)
lat = lmtl_biomass[Coordinates.Y.value]
lon = lmtl_biomass[Coordinates.X.value]
cell_areas = compute_spherical_cell_areas(lat, lon)
total_area = cell_areas.sum()

print(f"✅ Cell areas computed: {cell_areas.shape}")
print(f"Total domain area: {total_area.values / 1e12:.2f} × 10^12 m²")

# Display expected physics
print("\n" + "=" * 70)
print("EXPECTED NUMERICAL DIFFUSION CHARACTERISTICS")
print("=" * 70)
dx_mean = (lon.diff("x").mean() * 111000).values  # meters
u = 1.0  # m/s
dt_sim = dt_hours * 3600  # seconds
CFL = u * dt_sim / dx_mean
D_num_expected = (u * dx_mean / 2) * (1 - CFL**2)
D_phys = 3000  # m²/s

print(f"Grid spacing: {dx_mean:.0f} m")
print(f"Velocity: u = {u:.1f} m/s")
print(f"Timestep: dt = {dt_hours:.2f}h ({dt_sim:.0f}s)")
print(f"CFL: {CFL:.4f}")
print(f"D_num (expected): {D_num_expected:.0f} m²/s")
print(f"D_phys (configured): {D_phys:.0f} m²/s")

if D_phys > D_num_expected:
    print(f"✅ D_phys > D_num ({D_phys:.0f} > {D_num_expected:.0f})")
    print(
        f"   Margin: {D_phys - D_num_expected:.0f} m²/s ({(D_phys / D_num_expected - 1) * 100:.1f}%)"
    )
else:
    print(f"⚠️  D_num ≥ D_phys: Numerical diffusion may dominate!")
print("=" * 70)

In [ ]:
## MAPE / MPE ------------------------------------------------------------------------------------------------------------- ##
# Mean Absolute Percentage Error (MAPE) and Mean Percentage Error (MPE)
# These metrics quantify the relative differences between LMTL and SeapoPym

mape = np.abs((lmtl_biomass - seapopym_biomass) / (lmtl_biomass + 1e-10)) * 100
mape.attrs = {"units": "%"}

mpe = ((lmtl_biomass - seapopym_biomass) / (lmtl_biomass + 1e-10)) * 100
mpe.attrs = {"units": "%"}

# Visualization limits
max_err = 50  # Maximum error percentage for visualization

print(f"MAPE range: [{float(mape.min()):.2f}, {float(mape.max()):.2f}]%")
print(f"MPE range: [{float(mpe.min()):.2f}, {float(mpe.max()):.2f}]%")

In [ ]:
fig, axis = plt.subplots(1, 2, figsize=(18, 6))
mape.isel(time=slice(None, None)).mean("time").plot(vmax=max_err, cmap="viridis", ax=axis[0])
axis[0].set_title("MAPE (temporal mean)")
mpe.isel(time=slice(None, None)).mean("time").plot(vmax=max_err, cmap="RdYlGn_r", ax=axis[1])
axis[1].set_title("MPE (temporal mean)")

plt.tight_layout()
plt.show()

In [ ]:
fig, axis = plt.subplots(1, 2, figsize=(9, 3))

# ✅ CORRECTED: Conservation de masse avec pondération par surface
seapopym_total_mass = (seapopym_biomass * cell_areas).sum(["x", "y"])
lmtl_total_mass = (lmtl_biomass * cell_areas).sum(["x", "y"])

seapopym_growth_rate = (seapopym_total_mass / seapopym_total_mass.isel(time=0) * 100) - 100
seapopym_growth_rate.plot(ax=axis[0], label="Seapopym")

lmtl_growth_rate = (lmtl_total_mass / lmtl_total_mass.isel(time=0) * 100) - 100
lmtl_growth_rate.plot(ax=axis[0], label="LMTL")

axis[0].set_title("Total Mass Conservation (area-weighted)")
axis[0].set_xlabel("Time")
axis[0].set_ylabel("Variation (%)")
axis[0].grid()
axis[0].legend()

# ✅ CORRECTED: Moyenne spatiale de MAPE pondérée par surface
mape_spatial_mean = (mape * cell_areas).sum(["x", "y"]) / total_area
mape_spatial_mean.plot(ax=axis[1])
axis[1].set_title("MAPE (area-weighted spatial mean)")
axis[1].grid()
axis[1].set_xlabel("Time")
axis[1].set_ylabel("MAPE (%)")
axis[1].set_ylim(0, None)

plt.tight_layout()
plt.show()

In [ ]:
# Animation: MAPE/MPE evolution over time
# Calculer les moyennes hebdomadaires
mape_weekly = mape.resample(time=interval).mean()
mpe_weekly = mpe.resample(time=interval).mean()
lmtl_weekly = lmtl_biomass.resample(time=interval).mean()
seapopym_weekly = seapopym_biomass.resample(time=interval).mean()

seapopym_weekly = xr.where(seapopym_weekly.notnull(), seapopym_weekly, 0)
lmtl_weekly = xr.where(lmtl_weekly.notnull(), lmtl_weekly, 0)

# Créer la figure pour l'animation
fig = plt.figure(figsize=(18, 12))


def update(frame):
    """Fonction de mise à jour pour chaque frame de l'animation"""
    # Effacer complètement la figure
    fig.clear()

    # Recréer les subplots en grille 2x2
    ax1 = fig.add_subplot(2, 2, 1)
    ax2 = fig.add_subplot(2, 2, 2)
    ax3 = fig.add_subplot(2, 2, 3)
    ax4 = fig.add_subplot(2, 2, 4)

    # MAPE (haut gauche)
    mape_weekly.isel(time=frame).plot(vmax=max_err, cmap="viridis", ax=ax1)
    ax1.set_title(f"MAPE - Week {frame + 1}")

    # MPE (haut droite)
    mpe_weekly.isel(time=frame).plot(vmax=max_err, vmin=-max_err, cmap="RdYlGn_r", ax=ax2)
    ax2.set_title(f"MPE - Week {frame + 1}")

    # Biomasse LMTL (bas gauche)
    vmax_biomass = lmtl_weekly.max() / 100
    lmtl_weekly.isel(time=frame).plot(cmap="viridis", ax=ax3, vmax=vmax_biomass)
    ax3.set_title(f"Biomass LMTL - Week {frame + 1}")

    # Biomasse SeapoPym (bas droite)
    seapopym_weekly.isel(time=frame).plot(cmap="viridis", ax=ax4, vmax=vmax_biomass)
    ax4.set_title(f"Biomass SeapoPym - Week {frame + 1}")

    plt.tight_layout()


# Créer l'animation
n_frames = len(mape_weekly.time)
print(f"Creating animation with {n_frames} frames (weekly timesteps)...")
anim = FuncAnimation(fig, update, frames=n_frames, interval=500, repeat=True)

# Sauvegarder l'animation
print("Saving animation to mape_mpe_animation.gif...")
_ = anim.save("mape_mpe_animation.gif", writer="pillow", fps=10)
print("✅ Animation saved")

# RMSE / NRMSE

Root Mean Square Error (RMSE) and Normalized RMSE (NRMSE) provide absolute and relative measures of model differences.

In [ ]:
## RMSE ------------------------------------------------------------------------------------------------------------- ##
# ✅ CORRECTED: RMSE pondéré par surface des cellules

# RMSE pondéré par surface (métrique globale temporelle)
squared_error = (lmtl_biomass - seapopym_biomass) ** 2
weighted_squared_error = squared_error * cell_areas
rmse_global = np.sqrt(weighted_squared_error.sum(["x", "y"]) / total_area)
rmse_global.attrs = {"units": "g/m²"}

# RMSE spatial (2D) pour visualisation
rmse_spatial = np.sqrt(squared_error.mean("time"))
rmse_spatial.attrs = {"units": "g/m²"}

# NRMSE avec normalisation pondérée
weighted_biomass = lmtl_biomass * cell_areas
mean_weighted = weighted_biomass.sum(["x", "y"]) / total_area
variance_weighted = (((lmtl_biomass - mean_weighted) ** 2) * cell_areas).sum(
    ["x", "y"]
) / total_area
std_weighted = np.sqrt(variance_weighted)

nrmse_global = rmse_global / std_weighted
nrmse_global.attrs = {"units": "dimensionless"}

# NRMSE spatial pour visualisation
nrmse_spatial = rmse_spatial / lmtl_biomass.std(dim="time")
nrmse_spatial.attrs = {"units": "dimensionless"}

vmin = 0
vmax_rmse = float(rmse_spatial.quantile(0.95))  # Use 95th percentile for better visualization
vmax_nrmse = 1.0

print(f"Global RMSE (time-averaged): {rmse_global.mean().values:.4e} g/m²")
print(f"Global NRMSE (time-averaged): {nrmse_global.mean().values:.4f}")

In [ ]:
fig, axis = plt.subplots(1, 2, figsize=(18, 6))

# Plot spatial RMSE (averaged over time)
rmse_spatial.plot(cmap="viridis", ax=axis[0], vmin=vmin, vmax=vmax_rmse)
axis[0].set_title("RMSE (temporal mean)")

# Plot spatial NRMSE (averaged over time)
nrmse_spatial.plot(cmap="RdYlGn_r", ax=axis[1], vmin=vmin, vmax=vmax_nrmse)
axis[1].set_title("NRMSE (temporal mean)")

plt.tight_layout()
plt.show()

In [ ]:
fig, axis = plt.subplots(1, 2, figsize=(9, 3))

# ✅ Mass conservation (already computed with area-weighting)
seapopym_growth_rate.plot(ax=axis[0], label="Seapopym")
lmtl_growth_rate.plot(ax=axis[0], label="LMTL")

axis[0].set_title("Total Mass Conservation (area-weighted)")
axis[0].set_xlabel("Time")
axis[0].set_ylabel("Variation (%)")
axis[0].grid()
axis[0].legend()

# ✅ CORRECTED: Global NRMSE (area-weighted) over time
nrmse_global.plot(ax=axis[1])
axis[1].set_title("Global NRMSE (area-weighted)")
axis[1].grid()
axis[1].set_xlabel("Time")
axis[1].set_ylabel("NRMSE")
axis[1].set_ylim(0, None)

plt.tight_layout()
plt.show()

In [ ]:
# Animation: RMSE/NRMSE evolution over time
# Calculate weekly averages for biomass fields
lmtl_weekly = lmtl_biomass.resample(time=interval).mean()
seapopym_weekly = seapopym_biomass.resample(time=interval).mean()

seapopym_weekly = xr.where(seapopym_weekly.notnull(), seapopym_weekly, 0)
lmtl_weekly = xr.where(lmtl_weekly.notnull(), lmtl_weekly, 0)

# Calculate instantaneous RMSE and NRMSE for each weekly timestep
squared_error_weekly = (lmtl_weekly - seapopym_weekly) ** 2
rmse_weekly = np.sqrt(squared_error_weekly)
rmse_weekly.attrs = {"units": "g/m²"}

# NRMSE weekly (normalized by temporal std of LMTL)
lmtl_std = lmtl_weekly.std(dim="time")
nrmse_weekly = rmse_weekly / (lmtl_std + 1e-10)
nrmse_weekly.attrs = {"units": "dimensionless"}

# Créer la figure pour l'animation
fig = plt.figure(figsize=(18, 12))


def update(frame):
    """Fonction de mise à jour pour chaque frame de l'animation"""
    # Effacer complètement la figure
    fig.clear()

    # Recréer les subplots en grille 2x2
    ax1 = fig.add_subplot(2, 2, 1)
    ax2 = fig.add_subplot(2, 2, 2)
    ax3 = fig.add_subplot(2, 2, 3)
    ax4 = fig.add_subplot(2, 2, 4)

    # RMSE instantané (haut gauche)
    rmse_weekly.isel(time=frame).plot(cmap="viridis", ax=ax1, vmax=vmax_rmse, vmin=vmin)
    ax1.set_title(f"RMSE - Week {frame + 1}")

    # NRMSE instantané (haut droite)
    nrmse_weekly.isel(time=frame).plot(cmap="RdYlGn_r", ax=ax2, vmax=vmax_nrmse, vmin=vmin)
    ax2.set_title(f"NRMSE - Week {frame + 1}")

    # Biomasse LMTL (bas gauche)
    vmax_biomass = lmtl_weekly.max() / 2
    lmtl_weekly.isel(time=frame).plot(cmap="viridis", ax=ax3, vmax=vmax_biomass)
    ax3.set_title(f"Biomass LMTL - Week {frame + 1}")

    # Biomasse SeapoPym (bas droite)
    seapopym_weekly.isel(time=frame).plot(cmap="viridis", ax=ax4, vmax=vmax_biomass)
    ax4.set_title(f"Biomass SeapoPym - Week {frame + 1}")

    plt.tight_layout()


# Créer l'animation
n_frames = len(lmtl_weekly.time)
print(f"Creating animation with {n_frames} frames (weekly timesteps)...")
anim = FuncAnimation(fig, update, frames=n_frames, interval=500, repeat=True)

# Sauvegarder l'animation
print("Saving animation to rmse_nrmse_animation.gif...")
_ = anim.save("rmse_nrmse_animation.gif", writer="pillow", fps=10)
print("✅ Animation saved")